In [ ]:
import time
global_end_time = time.time() + 12 * 3600 - 20 * 60

# ARC Prize 2026 — NVARC half-cost double TTT

Fork of Ivan Sorokin's ARC Prize 2025 notebook (`sorokin/qwen3_4b_grids15_sft139`). Two half-cost LoRA TTT passes (8 train augs / 8 decode views, different seeds), **cheap-first both**. After pass A, `submission.json` is written immediately (A top-2). Pass B pools in with **keep-primary** (A top-1 always one of the two attempts) and rewrites the file every ~90s.

Local 120 public-eval: full-cost 28.19%; half-A 30.00%; half A+B pool 33.61%. Hidden v5 full-cost 28.47; v6 two-pass without A floor 28.75. This version is the timeout-safe half-cost recipe, not a guarantee of 33 on the LB.


In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_paths.py
import os
from pathlib import Path

COMP_NAMES = [
    "arc-prize-2026-arc-agi-2",
    "arc-prize-2025",
]
COMP_ROOTS = []
for name in COMP_NAMES:
    COMP_ROOTS.extend([
        Path("/kaggle/input/competitions") / name,
        Path("/kaggle/input") / name,
    ])


def competition_file(filename: str) -> str:
    for root in COMP_ROOTS:
        p = root / filename
        if p.exists():
            return str(p)
    hits = list(Path("/kaggle/input").rglob(filename))
    if hits:
        return str(hits[0])
    raise FileNotFoundError(filename)


def nvarc_model_dir() -> str:
    candidates = [
        Path("/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
        Path("/kaggle/input/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
        Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"),
        Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1"),
    ]
    for c in candidates:
        if (c / "config.json").exists():
            return str(c)
    for p in Path("/kaggle/input").rglob("config.json"):
        s = str(p).lower()
        if "grids15" in s or "qwen3_4b" in s:
            return str(p.parent)
    raise FileNotFoundError("NVARC model dir")


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        if not store or not os.path.isdir(store):
            return
        for key in os.listdir(store):
            path = os.path.join(store, key)
            if not os.path.isfile(path):
                continue
            try:
                with bz2.BZ2File(path) as f:
                    outputs = pickle.load(f)
            except Exception as e:
                print(f"skip pickle {key}: {e}", flush=True)
                continue
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import zlib
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

# 0 disables the cap. Kaggle notebook keeps the original 540 / 1200 constants;
# local runs set these via env (see run_local.sh --no-timeouts).
def _env_limit(name, default):
    raw = os.getenv(name)
    if raw is None or raw == "":
        return float(default)
    return float(raw)


def _env(name, default, cast):
    v = os.getenv(name)
    return cast(v) if v not in (None, "") else default


def stable_seed(key, offset=0):
    return (zlib.crc32(str(key).encode("utf-8")) + offset) % (1024 ** 2)


def _within_dfs_budget(start_time, end_time) -> bool:
    dfs_limit = _env_limit("NVARC_DFS_LIMIT", 540)
    if dfs_limit > 0 and time.time() - start_time >= dfs_limit:
        return False
    if end_time > 0 and time.time() >= end_time:
        return False
    return True


def _task_timed_out(start_time, end_time) -> bool:
    task_limit = _env_limit("NVARC_TASK_LIMIT", 1200)
    if task_limit > 0 and time.time() - start_time >= task_limit:
        return True
    if end_time > 0 and time.time() >= end_time:
        return True
    return False


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while _within_dfs_budget(start_time, end_time):

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    lora_seed = _env("ARC_LORA_SEED", 42, int)
    train_aug_seed = _env("ARC_TRAIN_AUG_SEED", 1, int)
    n_train_aug = _env("ARC_N_TRAIN_AUG", 16, int)
    eval_aug_seed = _env("ARC_EVAL_AUG_SEED", 2, int)
    n_eval_aug = _env("ARC_N_EVAL_AUG", 2, int)
    n_eval_geos = _env("ARC_N_EVAL_GEOS", 8, int)
    score_seed_off = _env("ARC_SCORE_SEED_OFFSET", 0, int)
    print(
        f"[Rank {rank}] seeds lora={lora_seed} train_aug={train_aug_seed} n={n_train_aug} "
        f"eval_aug={eval_aug_seed} n={n_eval_aug} geos={n_eval_geos} score_off={score_seed_off}"
    )

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=lora_seed,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=lora_seed,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    # Local port: NVARC_MODEL. Kaggle notebook: arc_paths.nvarc_model_dir().
    model_path = os.getenv("NVARC_MODEL")
    if not model_path:
        try:
            from arc_paths import nvarc_model_dir
            model_path = nvarc_model_dir()
        except Exception:
            model_path = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    test_path = os.getenv("NVARC_DATA")
    if not test_path:
        if rerun_mode:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        else:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = os.getenv("NVARC_OUT", "/kaggle/inference_outputs")
    os.makedirs(dir_outputs, exist_ok=True)
    print(f"[Rank {rank}] model={model_path} data={test_path} out={dir_outputs}")

    while not queue.empty():

        if end_time > 0 and time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=n_train_aug, shfl_keys=True, seed=train_aug_seed)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=n_eval_aug, seed=eval_aug_seed)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        # Batched DFS needs equal prefix lengths. rot0/rot180 share H×W; rot90/rot270 share W×H.
        n_perm = n_eval_aug
        batch_size = 4
        pairs = [(0, 2), (1, 3)]
        singles = []
        if n_eval_geos >= 8:
            pairs += [(4, 6), (5, 7)]
        elif n_eval_geos >= 6:
            pairs += [(4, 6)]
        elif n_eval_geos >= 5:
            singles = [4]
        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            for a, b in pairs:
                views = subkeys[a * n_perm:(a + 1) * n_perm] + subkeys[b * n_perm:(b + 1) * n_perm]
                if n_perm == 2 and batch_size == 4:
                    batches.append(views)
                else:
                    for i in range(0, len(views), batch_size):
                        batches.append(views[i:i + batch_size])
            for a in singles:
                views = subkeys[a * n_perm:(a + 1) * n_perm]
                if views:
                    batches.append(views)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if _task_timed_out(start_time, end_time):
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=stable_seed(bk, score_seed_off))
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import sys
import json
import time
import hashlib
import argparse
import signal
import threading
import subprocess
import torch
import torch.multiprocessing as mp
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
from arc_paths import competition_file, nvarc_model_dir

PUBLIC_TEST_KEYHASH = "5c5f0dce4b166234"


def keyhash(data):
    return hashlib.sha1("".join(sorted(data)).encode()).hexdigest()[:16]


def estimated_work(task):
    def ntok(g):
        return len(g) * (len(g[0]) + 1) if g and g[0] else 0
    train_tokens = sum(ntok(p["input"]) + ntok(p["output"]) for p in task["train"])
    ratios = [ntok(p["output"]) / max(1, ntok(p["input"])) for p in task["train"]] or [1.0]
    ratios.sort()
    ratio = ratios[len(ratios) // 2]
    test_tokens = sum(ntok(t["input"]) * (1 + ratio) for t in task["test"])
    return train_tokens * 8 + test_tokens * 8 * len(task["test"])


def live_checkpoint():
    script = "/kaggle/working/nvarc_checkpoint.py"
    primary = os.getenv("NVARC_CHECKPOINT_PRIMARY", "")
    if not primary or not os.path.exists(script):
        return
    data = os.getenv("NVARC_DATA") or ""
    sub = os.getenv("NVARC_CHECKPOINT_SUB", "/kaggle/working/submission.json")
    cmd = [sys.executable, script, "--outputs", primary, "--submission", sub, "--keep-primary"]
    if data:
        cmd.extend(["--data", data])
    extras = [p for p in os.getenv("NVARC_CHECKPOINT_EXTRAS", "").split(":") if p]
    out = os.getenv("NVARC_OUT", "")
    if out and out != primary and out not in extras:
        extras.append(out)
    for extra in extras:
        cmd.extend(["--outputs-extra", extra])
    try:
        subprocess.call(cmd)
    except Exception as e:
        print("live checkpoint:", e, flush=True)


def local_worker(rank, queue, end_time, sync_dir):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    torch.set_default_device("cpu")

    if rank > 0:
        waited = 0
        marker = os.path.join(sync_dir, f"worker{rank-1}")
        while not os.path.exists(marker) and waited < 900:
            time.sleep(5)
            waited += 5

    from arc_solver import worker

    with open(os.path.join(sync_dir, f"worker{rank}"), "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!", flush=True)
    worker(rank, queue, end_time)
    print(f"[Rank {rank}] done!", flush=True)


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    parser.add_argument("--out", default="/kaggle/inference_outputs")
    parser.add_argument("--keys-file", default="")
    parser.add_argument("--skip-done", action="store_true")
    parser.add_argument("--order", default="cheap", choices=["cheap", "sorted", "expensive"])
    args = parser.parse_args()

    rerun_env = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
    test_path = competition_file("arc-agi_test_challenges.json")
    with open(test_path, "r") as f:
        test_data = json.load(f)
    hidden = rerun_env or keyhash(test_data) != PUBLIC_TEST_KEYHASH

    if hidden:
        data_path = test_path
        data = test_data
        keys = list(data.keys())
    else:
        data_path = competition_file("arc-agi_evaluation_challenges.json")
        with open(data_path, "r") as f:
            data = json.load(f)
        keys = sorted(data.keys())[:4]

    if args.keys_file:
        wanted = json.load(open(args.keys_file)) if args.keys_file.endswith(".json") else [
            l.strip() for l in open(args.keys_file) if l.strip()
        ]
        keys = [k for k in wanted if k in data]

    if args.order == "cheap":
        keys = sorted(keys, key=lambda k: estimated_work(data[k]))
    elif args.order == "expensive":
        keys = sorted(keys, key=lambda k: estimated_work(data[k]), reverse=True)
    else:
        keys = sorted(keys)

    os.makedirs(args.out, exist_ok=True)
    if args.skip_done:
        done = {fn.split("_", 1)[0] for fn in os.listdir(args.out) if "_" in fn}
        keys = [k for k in keys if k not in done]

    nprocs = min(4, max(1, torch.cuda.device_count()))
    cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
    skipped = cap[0] < 7 and not hidden
    gate = {
        "python": sys.version.split()[0],
        "rerun_env": rerun_env,
        "hidden": hidden,
        "test_keyhash": keyhash(test_data),
        "n_test": len(test_data),
        "path": data_path,
        "nkeys_queued": 0 if skipped else len(keys),
        "nprocs": nprocs,
        "cap": list(cap),
        "skipped_ttt": skipped,
        "out": args.out,
    }
    Path("/kaggle/working/gate.json").write_text(json.dumps(gate))
    print(
        f"GATE starter python={gate['python']} cap=sm_{cap[0]}{cap[1]} nprocs={nprocs} "
        f"hidden={hidden} rerun_env={rerun_env} n_test={len(test_data)} queued={gate['nkeys_queued']} "
        f"path={data_path} out={args.out}",
        flush=True,
    )

    if skipped:
        print("skip TTT on sm_<70 Save Version (not hidden rerun)", flush=True)
        raise SystemExit(0)
    if cap[0] < 7:
        raise SystemExit("GATE FAIL: hidden rerun on sm_<70, refusing dummy submission")

    os.environ["NVARC_MODEL"] = os.getenv("NVARC_MODEL") or nvarc_model_dir()
    os.environ["NVARC_DATA"] = data_path
    os.environ["NVARC_OUT"] = args.out

    sync_dir = args.out.rstrip("/") + ".sync"
    os.makedirs(sync_dir, exist_ok=True)
    for fn in os.listdir(sync_dir):
        os.remove(os.path.join(sync_dir, fn))

    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    print(
        f"starter: nprocs={nprocs} nkeys={len(keys)} order={args.order} "
        f"budget_h={(args.end_time - time.time())/3600:.2f} out={args.out}",
        flush=True,
    )

    stop = threading.Event()

    def _on_term(signum, frame):
        print(f"starter signal {signum}: checkpoint then stop", flush=True)
        try:
            live_checkpoint()
        except Exception as e:
            print("signal checkpoint:", e, flush=True)

    signal.signal(signal.SIGTERM, _on_term)
    signal.signal(signal.SIGINT, _on_term)

    def _loop():
        interval = float(os.getenv("NVARC_CHECKPOINT_EVERY", "90"))
        while not stop.wait(interval):
            live_checkpoint()

    t = threading.Thread(target=_loop, daemon=True)
    t.start()
    try:
        live_checkpoint()
        mp.spawn(local_worker, args=(queue, args.end_time, sync_dir), nprocs=nprocs)
    finally:
        stop.set()
        live_checkpoint()


In [ ]:
%%writefile nvarc_checkpoint.py
"""Atomically write submission.json from pickle dirs.

Pass-A is --outputs (keep-primary). Later passes are --outputs-extra.
A hard kill cannot tear the JSON: we fsync a temp file, then os.replace.

Does not overwrite an existing scored file with an all-[[0]] result.
"""
from __future__ import annotations

import argparse
import json
import os
from pathlib import Path

import numpy as np

from arc_decoder import ArcDecoder, score_kgmon
from arc_loader import ArcDataset


def merge_keep_primary(sel_a, sel_p):
    """Pooled ranking, but pass-A top-1 is always one of the two attempts."""
    selected = {}
    n_forced = 0
    for bk in set(sel_a) | set(sel_p):
        a1 = (sel_a.get(bk) or [None])[0]
        top = list(sel_p.get(bk) or [])[:2]
        if a1 is not None and not any(np.array_equal(a1, g) for g in top):
            top = (top[:1] + [a1]) if top else [a1]
            n_forced += 1
        selected[bk] = top
    print(f"keep-primary: forced pass-A top-1 back on {n_forced} outputs", flush=True)
    return selected


def n_nonzero(submission) -> int:
    n = 0
    for attempts in submission.values():
        for a in attempts:
            for g in a.values():
                if g != [[0]]:
                    n += 1
    return n


def atomic_write_json(path: str | Path, obj) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    payload = json.dumps(obj)
    with open(tmp, "w") as f:
        f.write(payload)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def build_submission(data_path: str, primary: str, extras: list[str], keep_primary: bool):
    data = ArcDataset.from_file(data_path)
    decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
    if not os.path.isdir(primary):
        raise SystemExit(f"primary pickle dir missing: {primary}")
    decoder.load_decoded_results(primary)
    n_primary = sum(len(v) for v in decoder.decoded_results.values())
    sel_primary = decoder.run_selection_algo(score_kgmon) if keep_primary else None
    n_extra = 0
    for i, extra in enumerate(extras, 1):
        if not extra or not os.path.isdir(extra):
            continue
        if not any(Path(extra).iterdir()):
            continue
        before = sum(len(v) for v in decoder.decoded_results.values())
        decoder.load_decoded_results(extra, run_name=f".p{i}")
        n_extra += sum(len(v) for v in decoder.decoded_results.values()) - before
    selected = decoder.run_selection_algo(score_kgmon) if decoder.decoded_results else None
    if sel_primary is not None and selected is not None:
        selected = merge_keep_primary(sel_primary, selected)
    submission = data.get_submission(selected)
    n_decoded = len(decoder.decoded_results)
    return submission, n_decoded, n_primary, n_extra


def maybe_write(submission, dest: str) -> bool:
    dest = str(dest)
    new_n = n_nonzero(submission)
    if os.path.exists(dest):
        try:
            old = json.loads(Path(dest).read_text())
            old_n = n_nonzero(old)
        except Exception:
            old_n = 0
        if new_n == 0 and old_n > 0:
            print(f"checkpoint skip: new has 0 nonzero, keep existing {old_n}", flush=True)
            return False
    atomic_write_json(dest, submission)
    print(f"checkpoint wrote {dest} tasks={len(submission)} nonzero={new_n}", flush=True)
    return True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="", help="challenges json; inferred from gate.json if omitted")
    ap.add_argument("--outputs", required=True, help="pass-A pickle dir (keep-primary source)")
    ap.add_argument("--outputs-extra", action="append", default=[], help="later-pass pickle dirs")
    ap.add_argument("--keep-primary", action="store_true",
                    help="force pass-A top-1 to remain among the two attempts")
    ap.add_argument("--submission", default="/kaggle/working/submission.json")
    args = ap.parse_args()

    data_path = args.data
    if not data_path:
        gp = Path("/kaggle/working/gate.json")
        if gp.exists():
            data_path = json.loads(gp.read_text()).get("path") or ""
    if not data_path:
        try:
            from arc_paths import competition_file
            hidden = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
            data_path = competition_file(
                "arc-agi_test_challenges.json" if hidden else "arc-agi_evaluation_challenges.json"
            )
        except Exception as e:
            raise SystemExit(f"need --data ({e})")

    submission, n_decoded, n_primary, n_extra = build_submission(
        data_path, args.outputs, args.outputs_extra, args.keep_primary,
    )
    print(
        f"checkpoint decoded={n_decoded} samples_a={n_primary} samples_extra={n_extra} "
        f"keep_primary={args.keep_primary}",
        flush=True,
    )
    if n_decoded == 0:
        print("checkpoint: nothing decoded yet", flush=True)
        return
    maybe_write(submission, args.submission)


if __name__ == "__main__":
    main()


In [ ]:
import os, sys, time, json, subprocess
from pathlib import Path

# half2: 8 train augs / 8 decode views, two cheap-first passes, keep-primary.
# Flip to "third3" only after local 5x5x3 pool beats half-cost 33.61%.
SCHEDULE = "half2"

COMMON = {
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
    "OMP_NUM_THREADS": "12",
    "PYTHONHASHSEED": "0",
    "TOKENIZERS_PARALLELISM": "false",
    "ARC_N_EVAL_AUG": "1",
    "NVARC_TASK_LIMIT": "800",
    "NVARC_DFS_LIMIT": "540",
    "NVARC_CHECKPOINT_EVERY": "90",
    "NVARC_CHECKPOINT_SUB": "/kaggle/working/submission.json",
}
if SCHEDULE == "third3":
    COMMON["ARC_N_TRAIN_AUG"] = "5"
    COMMON["ARC_N_EVAL_GEOS"] = "5"
else:
    COMMON["ARC_N_TRAIN_AUG"] = "8"
    COMMON["ARC_N_EVAL_GEOS"] = "8"
os.environ.update(COMMON)

SEED_A = dict(ARC_LORA_SEED=42, ARC_TRAIN_AUG_SEED=1, ARC_EVAL_AUG_SEED=2, ARC_SCORE_SEED_OFFSET=0)
SEED_B = dict(ARC_LORA_SEED=137, ARC_TRAIN_AUG_SEED=17, ARC_EVAL_AUG_SEED=29, ARC_SCORE_SEED_OFFSET=7)
SEED_C = dict(ARC_LORA_SEED=271, ARC_TRAIN_AUG_SEED=31, ARC_EVAL_AUG_SEED=41, ARC_SCORE_SEED_OFFSET=13)

# Local half-cost xor is mid-rank, not the expensive tail. Both passes stay cheap-first.
if SCHEDULE == "third3":
    PASSES = [
        dict(name="A", out="/kaggle/inference_outputs", seeds=SEED_A, order="cheap"),
        dict(name="B", out="/kaggle/inference_outputs_b", seeds=SEED_B, order="cheap"),
        dict(name="C", out="/kaggle/inference_outputs_c", seeds=SEED_C, order="cheap"),
    ]
    RESERVE_PER_LATER = 2.8 * 3600
else:
    PASSES = [
        dict(name="A", out="/kaggle/inference_outputs", seeds=SEED_A, order="cheap"),
        dict(name="B", out="/kaggle/inference_outputs_b", seeds=SEED_B, order="cheap"),
    ]
    RESERVE_PER_LATER = 3.5 * 3600

SUB = "/kaggle/working/submission.json"
PRIMARY = PASSES[0]["out"]


def remaining():
    return global_end_time - time.time()


def checkpoint(dirs, tag):
    cmd = [sys.executable, "nvarc_checkpoint.py", "--outputs", dirs[0], "--submission", SUB, "--keep-primary"]
    gp = Path("/kaggle/working/gate.json")
    if gp.exists():
        data_path = json.loads(gp.read_text()).get("path")
        if data_path:
            cmd.extend(["--data", data_path])
    for extra in dirs[1:]:
        if Path(extra).exists() and any(Path(extra).iterdir()):
            cmd.extend(["--outputs-extra", extra])
    print(f"[checkpoint {tag}] {' '.join(cmd)}", flush=True)
    rc = subprocess.call(cmd)
    print(f"[checkpoint {tag}] rc={rc} remaining={remaining()/60:.1f} min exists={Path(SUB).exists()}", flush=True)
    return rc


def run_pass(spec, end_time, pool_dirs):
    name, out_dir = spec["name"], spec["out"]
    if remaining() < 15 * 60:
        print(f"[pass {name}] skipped, remaining={remaining()/60:.1f} min", flush=True)
        return 0
    env = dict(os.environ)
    env.update({k: str(v) for k, v in spec["seeds"].items()})
    env["NVARC_CHECKPOINT_PRIMARY"] = PRIMARY
    extras = [d for d in pool_dirs if d != PRIMARY]
    env["NVARC_CHECKPOINT_EXTRAS"] = ":".join(extras)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "starter.py",
        "--end-time", str(end_time),
        "--out", out_dir,
        "--order", spec["order"],
    ]
    print(f"[pass {name}] start remaining={remaining()/60:.1f} min out={out_dir} order={spec['order']} extra={spec['seeds']}", flush=True)
    t0 = time.time()
    rc = subprocess.call(cmd, env=env)
    nfiles = len(list(Path(out_dir).glob("*"))) if Path(out_dir).exists() else 0
    print(f"[pass {name}] rc={rc} took={(time.time()-t0)/60:.1f} min files={nfiles} remaining={remaining()/60:.1f} min", flush=True)
    print(f"starter_exit={rc}", flush=True)
    return rc


done_dirs = []
for i, spec in enumerate(PASSES):
    later = len(PASSES) - i - 1
    end_time = global_end_time - later * RESERVE_PER_LATER
    if end_time < time.time() + 20 * 60:
        end_time = global_end_time
    pool_dirs = done_dirs + [spec["out"]]
    rc = run_pass(spec, end_time, pool_dirs)
    done_dirs.append(spec["out"])
    checkpoint(done_dirs, f"after-{spec['name']}")
    print(f"pass-{spec['name']} leftover files={len(list(Path(spec['out']).glob('*'))) if Path(spec['out']).exists() else 0}", flush=True)

print(f"passes done remaining={remaining()/60:.1f} min sub={Path(SUB).exists()}", flush=True)


In [ ]:
import sys, os, json, subprocess
from pathlib import Path

for key in list(sys.path):
    if "unsloth" in key.lower() or "flash_patch" in key.lower() or "/kaggle/usr/lib/notebooks" in key:
        try:
            sys.path.remove(key)
        except ValueError:
            pass
sys.path.insert(0, "/kaggle/working")

dirs = []
for p in ("/kaggle/inference_outputs", "/kaggle/inference_outputs_b", "/kaggle/inference_outputs_c"):
    if Path(p).exists() and any(Path(p).iterdir()):
        dirs.append(p)
if dirs:
    cmd = [sys.executable, "nvarc_checkpoint.py", "--outputs", dirs[0], "--submission", "/kaggle/working/submission.json", "--keep-primary"]
    gp = Path("/kaggle/working/gate.json")
    if gp.exists():
        data_path = json.loads(gp.read_text()).get("path")
        if data_path:
            cmd.extend(["--data", data_path])
    for extra in dirs[1:]:
        cmd.extend(["--outputs-extra", extra])
    print("final", " ".join(cmd), flush=True)
    subprocess.call(cmd)

gate = {}
if Path("/kaggle/working/gate.json").exists():
    gate = json.loads(Path("/kaggle/working/gate.json").read_text())
hidden = bool(gate.get("hidden") or os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
skipped = bool(gate.get("skipped_ttt"))
sub_path = Path("/kaggle/working/submission.json")
n_nonzero = 0
n_tasks = 0
if sub_path.exists():
    try:
        submission = json.loads(sub_path.read_text())
        n_tasks = len(submission)
        for attempts in submission.values():
            for a in attempts:
                for g in a.values():
                    if g != [[0]]:
                        n_nonzero += 1
    except Exception as e:
        print("read submission.json:", e, flush=True)

print(
    f"GATE last hidden={hidden} skipped_ttt={skipped} decoded={n_tasks} "
    f"nonzero={n_nonzero} path={gate.get('path')} python={gate.get('python')} cap={gate.get('cap')}",
    flush=True,
)
print("GATE nonzero_attempts", n_nonzero, "tasks", n_tasks, flush=True)

if not skipped and n_nonzero == 0:
    raise RuntimeError("GATE FAIL: TTT produced 0 decoded results; refusing dummy [[0]] submission")
if hidden and n_nonzero == 0:
    raise RuntimeError("GATE FAIL: hidden rerun submission is all [[0]]")

if not hidden and sub_path.exists():
    try:
        from arc_paths import competition_file
        from arc_loader import ArcDataset
        from arc_decoder import ArcDecoder, score_kgmon
        data = ArcDataset.from_file(gate.get("path") or competition_file("arc-agi_evaluation_challenges.json"))
        data = data.load_replies(competition_file("arc-agi_evaluation_solutions.json"))
        decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
        if dirs:
            decoder.load_decoded_results(dirs[0])
            for i, extra in enumerate(dirs[1:], 1):
                decoder.load_decoded_results(extra, run_name=f".p{i}")
        if decoder.decoded_results:
            decoder.benchmark_selection_algos()
        print("*** Reload score:", data.validate_submission(json.loads(sub_path.read_text())))
    except Exception as e:
        print("benchmark skipped:", e)
